# Классификация изображений с использованием SVM

Этот ноутбук реализует классификацию изображений с использованием метода опорных векторов (SVM).

**Датасет:** 1638 изображений, 6 классов, ~273 изображения на класс

**Структура:**
1. Подключение Google Drive
2. Загрузка и предобработка данных
3. Извлечение признаков
4. Обучение модели SVM
5. Оценка качества и визуализация
6. Сохранение модели
7. Интерфейс для предсказаний

## 1. Установка и импорт библиотек

In [ ]:
# Установка необходимых библиотек
!pip install scikit-learn opencv-python pillow matplotlib seaborn joblib tqdm -q

In [ ]:
# Импорт основных библиотек
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from tqdm import tqdm
import joblib
import warnings
warnings.filterwarnings('ignore')

# Импорт для машинного обучения
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, learning_curve
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize

# Настройка отображения графиков
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Все библиотеки успешно импортированы")

## 2. Подключение Google Drive

In [ ]:
# Подключение Google Drive для загрузки данных и сохранения моделей
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive успешно подключен")

In [ ]:
# НАСТРОЙКА ПУТЕЙ
# Укажите путь к папке с датасетом на вашем Google Drive
# Структура: dataset_folder/class_name/image.jpg
DATASET_PATH = '/content/drive/MyDrive/dataset'  # ИЗМЕНИТЕ НА ВАШ ПУТЬ
MODEL_SAVE_PATH = '/content/drive/MyDrive/models'  # Путь для сохранения моделей

# Создание папки для моделей, если её нет
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"Путь к датасету: {DATASET_PATH}")
print(f"Путь для сохранения моделей: {MODEL_SAVE_PATH}")

## 3. Загрузка и предобработка данных

In [ ]:
# Параметры предобработки
IMG_SIZE = (128, 128)  # Целевой размер изображений
COLOR_MODE = 'RGB'      # Цветовой режим (RGB или GRAY)

def load_images_from_folders(dataset_path, img_size=IMG_SIZE, color_mode=COLOR_MODE):
    """
    Загрузка изображений из структуры каталогов.
    
    Параметры:
    - dataset_path: путь к корневой папке датасета
    - img_size: целевой размер изображений (ширина, высота)
    - color_mode: 'RGB' или 'GRAY'
    
    Возвращает:
    - images: массив изображений
    - labels: массив меток классов
    - class_names: список имен классов
    """
    images = []
    labels = []
    class_names = []
    
    # Получение списка классов (папок)
    class_folders = [f for f in os.listdir(dataset_path) 
                     if os.path.isdir(os.path.join(dataset_path, f))]
    class_folders.sort()  # Сортировка для консистентности
    
    print(f"Найдено классов: {len(class_folders)}")
    print(f"Классы: {class_folders}\n")
    
    # Статистика по размерам изображений
    size_stats = {}
    
    # Загрузка изображений по классам
    for class_idx, class_name in enumerate(class_folders):
        class_path = os.path.join(dataset_path, class_name)
        class_names.append(class_name)
        
        # Получение списка файлов изображений
        image_files = [f for f in os.listdir(class_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        print(f"Загрузка класса '{class_name}': {len(image_files)} изображений")
        
        # Загрузка каждого изображения
        for img_file in tqdm(image_files, desc=f"  {class_name}"):
            img_path = os.path.join(class_path, img_file)
            
            try:
                # Загрузка изображения
                img = Image.open(img_path)
                
                # Сохранение статистики по размерам
                orig_size = f"{img.size[0]}x{img.size[1]}"
                size_stats[orig_size] = size_stats.get(orig_size, 0) + 1
                
                # Конвертация в нужный цветовой режим
                if color_mode == 'RGB':
                    img = img.convert('RGB')
                elif color_mode == 'GRAY':
                    img = img.convert('L')
                
                # Изменение размера
                img = img.resize(img_size, Image.Resampling.LANCZOS)
                
                # Преобразование в массив numpy
                img_array = np.array(img)
                
                images.append(img_array)
                labels.append(class_idx)
                
            except Exception as e:
                print(f"  ⚠ Ошибка загрузки {img_file}: {e}")
                continue
    
    # Вывод статистики
    print("\n" + "="*50)
    print("СТАТИСТИКА ЗАГРУЗКИ")
    print("="*50)
    print(f"Всего загружено изображений: {len(images)}")
    print(f"Количество классов: {len(class_names)}")
    print(f"\nРаспределение оригинальных размеров:")
    for size, count in sorted(size_stats.items()):
        print(f"  {size}: {count} изображений")
    print(f"\nЦелевой размер: {img_size[0]}x{img_size[1]}")
    print("="*50)
    
    # Конвертация в numpy массивы
    images = np.array(images)
    labels = np.array(labels)
    
    return images, labels, class_names

In [ ]:
# Загрузка данных
print("Начало загрузки данных...\n")
X_images, y_labels, class_names = load_images_from_folders(DATASET_PATH)

print(f"\n✓ Данные успешно загружены!")
print(f"Форма массива изображений: {X_images.shape}")
print(f"Форма массива меток: {y_labels.shape}")
print(f"Классы: {class_names}")

In [ ]:
# Визуализация примеров изображений из каждого класса
def visualize_samples(images, labels, class_names, samples_per_class=3):
    """
    Визуализация примеров изображений из каждого класса.
    """
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, samples_per_class, 
                             figsize=(samples_per_class*3, n_classes*3))
    
    for class_idx, class_name in enumerate(class_names):
        # Получение индексов изображений данного класса
        class_indices = np.where(labels == class_idx)[0]
        # Случайный выбор нескольких примеров
        sample_indices = np.random.choice(class_indices, samples_per_class, replace=False)
        
        for i, idx in enumerate(sample_indices):
            ax = axes[class_idx, i] if n_classes > 1 else axes[i]
            ax.imshow(images[idx])
            ax.axis('off')
            if i == 0:
                ax.set_title(f'{class_name}', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.suptitle('Примеры изображений из каждого класса', 
                 fontsize=16, fontweight='bold', y=1.002)
    plt.show()

# Визуализация
visualize_samples(X_images, y_labels, class_names)

In [ ]:
# Анализ распределения классов
def plot_class_distribution(labels, class_names):
    """
    Визуализация распределения классов в датасете.
    """
    unique, counts = np.unique(labels, return_counts=True)
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(range(len(unique)), counts, color='skyblue', edgecolor='navy')
    plt.xlabel('Класс', fontsize=12)
    plt.ylabel('Количество изображений', fontsize=12)
    plt.title('Распределение классов в датасете', fontsize=14, fontweight='bold')
    plt.xticks(range(len(unique)), class_names, rotation=45, ha='right')
    
    # Добавление значений на столбцы
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Вывод статистики
    print("\nСтатистика по классам:")
    for idx, class_name in enumerate(class_names):
        print(f"  {class_name}: {counts[idx]} изображений ({counts[idx]/len(labels)*100:.1f}%)")

plot_class_distribution(y_labels, class_names)

## 4. Извлечение признаков и подготовка данных

In [ ]:
def preprocess_images(images):
    """
    Предобработка изображений:
    1. Нормализация значений пикселей (0-1)
    2. Преобразование в одномерный вектор (flatten)
    
    Параметры:
    - images: массив изображений
    
    Возвращает:
    - features: массив векторов признаков
    """
    print("Предобработка изображений...")
    
    # Нормализация: приведение значений пикселей к диапазону [0, 1]
    images_normalized = images.astype('float32') / 255.0
    
    # Flatten: преобразование каждого изображения в одномерный вектор
    # Из (n_samples, height, width, channels) в (n_samples, height*width*channels)
    features = images_normalized.reshape(images_normalized.shape[0], -1)
    
    print(f"✓ Форма матрицы признаков: {features.shape}")
    print(f"  Каждое изображение представлено вектором из {features.shape[1]} признаков")
    
    return features

# Извлечение признаков
X_features = preprocess_images(X_images)

In [ ]:
# Разделение данных на обучающую и тестовую выборки
TEST_SIZE = 0.2  # 20% данных для тестирования
RANDOM_STATE = 42  # Для воспроизводимости результатов

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_labels  # Сохранение пропорций классов
)

print("Разделение данных:")
print(f"  Обучающая выборка: {X_train.shape[0]} образцов")
print(f"  Тестовая выборка: {X_test.shape[0]} образцов")
print(f"  Размерность признаков: {X_train.shape[1]}")

In [ ]:
# Стандартизация признаков
# Важно для SVM: приведение всех признаков к одному масштабу
print("Стандартизация признаков...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Стандартизация завершена")
print(f"  Среднее значение (train): {X_train_scaled.mean():.6f}")
print(f"  Стандартное отклонение (train): {X_train_scaled.std():.6f}")

In [ ]:
# Снижение размерности с помощью PCA (Principal Component Analysis)
# PCA помогает уменьшить количество признаков, сохраняя наиболее важную информацию
# Это ускоряет обучение и может улучшить качество модели

# Параметр: сохраняем 95% дисперсии данных
PCA_VARIANCE = 0.95

print(f"Применение PCA (сохранение {PCA_VARIANCE*100}% дисперсии)...")

pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("✓ PCA завершен")
print(f"  Исходная размерность: {X_train_scaled.shape[1]}")
print(f"  Размерность после PCA: {X_train_pca.shape[1]}")
print(f"  Снижение размерности: {(1 - X_train_pca.shape[1]/X_train_scaled.shape[1])*100:.1f}%")
print(f"  Объясненная дисперсия: {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
# Визуализация объясненной дисперсии по компонентам
plt.figure(figsize=(12, 5))

# График 1: Объясненная дисперсия каждой компонентой
plt.subplot(1, 2, 1)
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
         pca.explained_variance_ratio_, 'bo-', linewidth=2)
plt.xlabel('Номер компоненты', fontsize=12)
plt.ylabel('Объясненная дисперсия', fontsize=12)
plt.title('Дисперсия по компонентам PCA', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# График 2: Кумулятивная объясненная дисперсия
plt.subplot(1, 2, 2)
cumsum = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cumsum) + 1), cumsum, 'ro-', linewidth=2)
plt.axhline(y=PCA_VARIANCE, color='g', linestyle='--', 
            label=f'Порог {PCA_VARIANCE*100}%')
plt.xlabel('Номер компоненты', fontsize=12)
plt.ylabel('Кумулятивная дисперсия', fontsize=12)
plt.title('Кумулятивная объясненная дисперсия', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Обучение модели SVM

In [ ]:
# Параметры для поиска по сетке (Grid Search)
# GridSearchCV автоматически найдет лучшие гиперпараметры модели

param_grid = {
    'C': [0.1, 1, 10, 100],              # Параметр регуляризации
    'kernel': ['rbf', 'linear'],          # Тип ядра
    'gamma': ['scale', 'auto', 0.001, 0.01]  # Коэффициент ядра для RBF
}

print("Параметры для поиска:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")
print(f"\nВсего комбинаций: {np.prod([len(v) for v in param_grid.values()])}")

In [ ]:
# Создание и обучение модели с Grid Search
print("\nНачало обучения модели SVM...\n")
print("⏳ Это может занять несколько минут...\n")

# Базовая модель SVM
svm_base = SVC(random_state=RANDOM_STATE, probability=True)  # probability=True для ROC-AUC

# Grid Search с кросс-валидацией
grid_search = GridSearchCV(
    estimator=svm_base,
    param_grid=param_grid,
    cv=5,                    # 5-fold кросс-валидация
    scoring='accuracy',       # Метрика оптимизации
    n_jobs=-1,               # Использовать все CPU
    verbose=2                # Подробный вывод
)

# Запуск обучения
grid_search.fit(X_train_pca, y_train)

print("\n" + "="*50)
print("✓ ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("="*50)
print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучшая точность (CV): {grid_search.best_score_:.4f}")
print("="*50)

In [ ]:
# Получение лучшей модели
best_svm = grid_search.best_estimator_

# Вывод топ-5 комбинаций параметров
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

print("\nТоп-5 комбинаций параметров:\n")
top5 = results_df[['params', 'mean_test_score', 'std_test_score']].head(5)
for idx, row in top5.iterrows():
    print(f"{row['params']}")
    print(f"  Accuracy: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})\n")

## 6. Оценка качества модели

In [ ]:
# Предсказания на тестовой выборке
print("Выполнение предсказаний на тестовой выборке...")

y_pred = best_svm.predict(X_test_pca)
y_pred_proba = best_svm.predict_proba(X_test_pca)  # Вероятности для ROC

print("✓ Предсказания выполнены")

In [ ]:
# Вычисление основных метрик
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("\n" + "="*50)
print("МЕТРИКИ КАЧЕСТВА НА ТЕСТОВОЙ ВЫБОРКЕ")
print("="*50)
print(f"Accuracy (Точность):  {accuracy:.4f}")
print(f"Precision (Precision): {precision:.4f}")
print(f"Recall (Полнота):     {recall:.4f}")
print(f"F1-Score:             {f1:.4f}")
print("="*50)

In [ ]:
# Детальный отчет по классам
print("\nДЕТАЛЬНЫЙ ОТЧЕТ ПО КЛАССАМ:\n")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

In [ ]:
# Визуализация матрицы ошибок (Confusion Matrix)
def plot_confusion_matrix(y_true, y_pred, class_names):
    """
    Визуализация матрицы ошибок.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                cbar_kws={'label': 'Количество образцов'})
    plt.xlabel('Предсказанный класс', fontsize=12)
    plt.ylabel('Истинный класс', fontsize=12)
    plt.title('Матрица ошибок (Confusion Matrix)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Нормализованная матрица ошибок (в процентах)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
                xticklabels=class_names,
                yticklabels=class_names,
                cbar_kws={'label': 'Доля образцов'})
    plt.xlabel('Предсказанный класс', fontsize=12)
    plt.ylabel('Истинный класс', fontsize=12)
    plt.title('Нормализованная матрица ошибок (%)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_test, y_pred, class_names)

In [ ]:
# Визуализация метрик по классам
def plot_metrics_per_class(y_true, y_pred, class_names):
    """
    Визуализация метрик (Precision, Recall, F1) для каждого класса.
    """
    # Вычисление метрик для каждого класса
    precision_per_class = precision_score(y_true, y_pred, average=None)
    recall_per_class = recall_score(y_true, y_pred, average=None)
    f1_per_class = f1_score(y_true, y_pred, average=None)
    
    x = np.arange(len(class_names))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width, precision_per_class, width, label='Precision', color='skyblue')
    bars2 = ax.bar(x, recall_per_class, width, label='Recall', color='lightcoral')
    bars3 = ax.bar(x + width, f1_per_class, width, label='F1-Score', color='lightgreen')
    
    ax.set_xlabel('Класс', fontsize=12)
    ax.set_ylabel('Значение метрики', fontsize=12)
    ax.set_title('Метрики по классам', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3, axis='y')
    
    # Добавление значений на столбцы
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.show()

plot_metrics_per_class(y_test, y_pred, class_names)

In [ ]:
# ROC-кривые и AUC для многоклассовой классификации
def plot_roc_curves(y_true, y_pred_proba, class_names):
    """
    Визуализация ROC-кривых для каждого класса (One-vs-Rest).
    """
    n_classes = len(class_names)
    
    # Бинаризация меток для многоклассовой классификации
    y_test_bin = label_binarize(y_true, classes=range(n_classes))
    
    # Вычисление ROC-кривой и AUC для каждого класса
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Вычисление микро-усредненной ROC-кривой
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_pred_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    
    # Визуализация
    plt.figure(figsize=(12, 8))
    
    # ROC-кривые для каждого класса
    colors = plt.cm.Set3(np.linspace(0, 1, n_classes))
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')
    
    # Микро-усредненная ROC-кривая
    plt.plot(fpr["micro"], tpr["micro"],
            label=f'Micro-average (AUC = {roc_auc["micro"]:.3f})',
            color='deeppink', linestyle=':', linewidth=3)
    
    # Диагональная линия (случайный классификатор)
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Случайный классификатор')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)', fontsize=12)
    plt.ylabel('True Positive Rate (TPR)', fontsize=12)
    plt.title('ROC-кривые (One-vs-Rest)', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Вывод AUC для каждого класса
    print("\nAUC (Area Under Curve) для каждого класса:")
    for i, class_name in enumerate(class_names):
        print(f"  {class_name}: {roc_auc[i]:.4f}")
    print(f"  Micro-average: {roc_auc['micro']:.4f}")

plot_roc_curves(y_test, y_pred_proba, class_names)

In [ ]:
# Кривые обучения (Learning Curves)
def plot_learning_curves(estimator, X, y, cv=5):
    """
    Визуализация кривых обучения для анализа переобучения/недообучения.
    """
    print("Вычисление кривых обучения...")
    print("⏳ Это может занять несколько минут...\n")
    
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y,
        cv=cv,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy',
        verbose=0
    )
    
    # Вычисление среднего и стандартного отклонения
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    # Визуализация
    plt.figure(figsize=(12, 6))
    
    plt.plot(train_sizes, train_mean, 'o-', color='r', label='Обучающая выборка')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                     alpha=0.1, color='r')
    
    plt.plot(train_sizes, val_mean, 'o-', color='g', label='Валидационная выборка')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                     alpha=0.1, color='g')
    
    plt.xlabel('Размер обучающей выборки', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('Кривые обучения (Learning Curves)', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("✓ Кривые обучения построены")
    print(f"\nАнализ:")
    if train_mean[-1] - val_mean[-1] > 0.1:
        print("  ⚠ Возможно переобучение (большой разрыв между train и validation)")
    elif val_mean[-1] < 0.7:
        print("  ⚠ Возможно недообучение (низкая точность на validation)")
    else:
        print("  ✓ Модель обучена хорошо")

# Построение кривых обучения для лучшей модели
plot_learning_curves(best_svm, X_train_pca, y_train)

## 7. Сохранение модели и компонентов

In [ ]:
# Сохранение модели и всех необходимых компонентов
print("Сохранение модели и компонентов...\n")

# Создание словаря с моделью и компонентами
model_package = {
    'model': best_svm,            # Обученная SVM модель
    'scaler': scaler,              # Scaler для стандартизации
    'pca': pca,                    # PCA для снижения размерности
    'class_names': class_names,    # Имена классов
    'img_size': IMG_SIZE,          # Размер изображений
    'color_mode': COLOR_MODE,      # Цветовой режим
    'best_params': grid_search.best_params_,  # Лучшие параметры
    'metrics': {                   # Метрики на тестовой выборке
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }
}

# Путь для сохранения
model_filename = os.path.join(MODEL_SAVE_PATH, 'svm_image_classifier.pkl')

# Сохранение
joblib.dump(model_package, model_filename)

print("✓ Модель сохранена успешно!")
print(f"  Файл: {model_filename}")
print(f"  Размер: {os.path.getsize(model_filename) / (1024*1024):.2f} MB")

## 8. Загрузка и использование сохраненной модели

In [ ]:
# Функция для загрузки модели
def load_model(model_path):
    """
    Загрузка сохраненной модели и компонентов.
    
    Параметры:
    - model_path: путь к файлу модели
    
    Возвращает:
    - model_package: словарь с моделью и компонентами
    """
    print(f"Загрузка модели из {model_path}...")
    model_package = joblib.load(model_path)
    print("✓ Модель загружена успешно!\n")
    
    print("Информация о модели:")
    print(f"  Классы: {model_package['class_names']}")
    print(f"  Размер изображений: {model_package['img_size']}")
    print(f"  Лучшие параметры: {model_package['best_params']}")
    print(f"\n  Метрики на тестовой выборке:")
    for metric, value in model_package['metrics'].items():
        print(f"    {metric}: {value:.4f}")
    
    return model_package

# Пример загрузки (раскомментируйте для использования)
# loaded_model = load_model(model_filename)

## 9. Интерфейс для предсказаний на новых изображениях

In [ ]:
def predict_image(image_path, model_package, show_probabilities=True):
    """
    Предсказание класса для одного изображения.
    
    Параметры:
    - image_path: путь к изображению
    - model_package: загруженная модель с компонентами
    - show_probabilities: показывать вероятности для всех классов
    
    Возвращает:
    - predicted_class: предсказанный класс
    - probability: вероятность предсказанного класса
    """
    # Извлечение компонентов
    model = model_package['model']
    scaler = model_package['scaler']
    pca = model_package['pca']
    class_names = model_package['class_names']
    img_size = model_package['img_size']
    color_mode = model_package['color_mode']
    
    # Загрузка и предобработка изображения
    img = Image.open(image_path)
    
    if color_mode == 'RGB':
        img = img.convert('RGB')
    elif color_mode == 'GRAY':
        img = img.convert('L')
    
    img = img.resize(img_size, Image.Resampling.LANCZOS)
    img_array = np.array(img)
    
    # Нормализация и flatten
    img_normalized = img_array.astype('float32') / 255.0
    img_flat = img_normalized.reshape(1, -1)
    
    # Применение scaler и PCA
    img_scaled = scaler.transform(img_flat)
    img_pca = pca.transform(img_scaled)
    
    # Предсказание
    prediction = model.predict(img_pca)[0]
    probabilities = model.predict_proba(img_pca)[0]
    
    predicted_class = class_names[prediction]
    probability = probabilities[prediction]
    
    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Изображение
    axes[0].imshow(img_array)
    axes[0].axis('off')
    axes[0].set_title(f'Предсказание: {predicted_class}\nВероятность: {probability:.2%}',
                     fontsize=14, fontweight='bold')
    
    # График вероятностей
    if show_probabilities:
        sorted_indices = np.argsort(probabilities)[::-1]
        sorted_classes = [class_names[i] for i in sorted_indices]
        sorted_probs = probabilities[sorted_indices]
        
        colors = ['green' if i == prediction else 'skyblue' for i in sorted_indices]
        axes[1].barh(sorted_classes, sorted_probs, color=colors)
        axes[1].set_xlabel('Вероятность', fontsize=12)
        axes[1].set_title('Вероятности классов', fontsize=14, fontweight='bold')
        axes[1].set_xlim([0, 1])
        
        # Добавление значений на столбцы
        for i, prob in enumerate(sorted_probs):
            axes[1].text(prob + 0.02, i, f'{prob:.2%}', 
                        va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    return predicted_class, probability

print("✓ Функция predict_image готова к использованию")

In [ ]:
# Пример использования функции предсказания
# Раскомментируйте и укажите путь к изображению для тестирования

# test_image_path = '/content/drive/MyDrive/test_image.jpg'  # УКАЖИТЕ ПУТЬ
# predicted_class, probability = predict_image(test_image_path, model_package)
# print(f"\nИтоговое предсказание: {predicted_class} ({probability:.2%})")

In [ ]:
# Массовое предсказание для папки с изображениями
def predict_folder(folder_path, model_package, limit=None):
    """
    Предсказание для всех изображений в папке.
    
    Параметры:
    - folder_path: путь к папке с изображениями
    - model_package: загруженная модель с компонентами
    - limit: максимальное количество изображений для обработки
    
    Возвращает:
    - results: DataFrame с результатами
    """
    # Получение списка файлов
    image_files = [f for f in os.listdir(folder_path) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if limit:
        image_files = image_files[:limit]
    
    print(f"Обработка {len(image_files)} изображений...\n")
    
    results = []
    
    for img_file in tqdm(image_files):
        img_path = os.path.join(folder_path, img_file)
        try:
            predicted_class, probability = predict_image(
                img_path, model_package, show_probabilities=False
            )
            plt.close()  # Закрыть график
            
            results.append({
                'filename': img_file,
                'predicted_class': predicted_class,
                'probability': probability
            })
        except Exception as e:
            print(f"Ошибка обработки {img_file}: {e}")
    
    results_df = pd.DataFrame(results)
    
    print("\n✓ Обработка завершена!")
    print(f"\nСтатистика предсказаний:")
    print(results_df['predicted_class'].value_counts())
    
    return results_df

# Пример использования
# test_folder = '/content/drive/MyDrive/test_images'  # УКАЖИТЕ ПУТЬ
# results_df = predict_folder(test_folder, model_package, limit=10)
# results_df.head()

## 10. Сводная информация о модели

In [ ]:
# Итоговая сводка
print("\n" + "="*60)
print(" "*15 + "ИТОГОВАЯ СВОДКА")
print("="*60)

print("\n📊 ДАННЫЕ:")
print(f"  • Всего изображений: {len(X_images)}")
print(f"  • Количество классов: {len(class_names)}")
print(f"  • Классы: {', '.join(class_names)}")
print(f"  • Размер изображений: {IMG_SIZE[0]}x{IMG_SIZE[1]}")
print(f"  • Обучающая выборка: {len(X_train)} ({len(X_train)/len(X_images)*100:.1f}%)")
print(f"  • Тестовая выборка: {len(X_test)} ({len(X_test)/len(X_images)*100:.1f}%)")

print("\n🤖 МОДЕЛЬ:")
print(f"  • Алгоритм: Support Vector Machine (SVM)")
print(f"  • Ядро: {grid_search.best_params_['kernel']}")
print(f"  • Параметры: {grid_search.best_params_}")
print(f"  • Размерность после PCA: {X_train_pca.shape[1]} ({X_train_pca.shape[1]/X_train.shape[1]*100:.1f}% от исходной)")

print("\n📈 КАЧЕСТВО:")
print(f"  • Accuracy:  {accuracy:.4f}")
print(f"  • Precision: {precision:.4f}")
print(f"  • Recall:    {recall:.4f}")
print(f"  • F1-Score:  {f1:.4f}")

print("\n💾 СОХРАНЕНИЕ:")
print(f"  • Файл модели: {model_filename}")
print(f"  • Размер: {os.path.getsize(model_filename) / (1024*1024):.2f} MB")

print("\n" + "="*60)
print("✓ Модель успешно обучена и готова к использованию!")
print("="*60)